# BAM 실험용 데이터셋 생성 (Clean + Noisy 저장)

- 목적: **클린 IQ(정규화, |x|≤1)**를 저장하고,
  SNR 범위 `0 ~ -30 dB`에서 **노이즈를 1~2회(설정 가능) 미리 생성**해 저장
- Training 시: 디스크의 `(noisy, clean)` 페어를 로드해서 단층 BAM이 MSE를 줄이는지 확인

## 출력 폴더 구조(예시)
- `dataset_.../clean_iq_norm/*.npy`  (클린)
- `dataset_.../noisy_iq_norm/*.npy`  (노이즈 추가된 noisy)
- `dataset_.../dataset_meta.json`

In [ ]:
import numpy as np
import os, sys, json
import importlib

# 상위 디렉토리의 utils 모듈 사용
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print("Loaded my_lora_utils from:", utils.my_lora_utils.__file__)

# =========================
# LoRa 파라미터 (기존 값)
# =========================
sf = 9
bw = 125_000
OSF = 8
fs = int(bw * OSF)

symbol_time = 2**sf / bw
N = 2**sf  # 512 symbols

print(f"SF={sf}, BW={bw/1e3:.1f}kHz, OSF={OSF}, fs={fs/1e6:.2f}MHz")
print(f"Samples/symbol = {2**sf * OSF}, #symbols={N}")

# =========================
# 데이터 생성 설정
# =========================
NOISY_PER_CLEAN = 10          # 1 또는 2 추천
SNR_DB_HIGH = 0.0            # 0 dB
SNR_DB_LOW  = -30.0          # -30 dB

# =========================
# LoRa 객체
# =========================
lora = LoRa(sf, bw, OSF)

# =========================
# 저장 경로 (Clean + Noisy)
# =========================
base_dir = f"dataset_sf{sf}_bw{int(bw/1e3)}k_osf{OSF}_fs{int(fs/1e6)}MHz"
clean_dir = os.path.join(base_dir, "clean_iq_norm")
noisy_dir = os.path.join(base_dir, "noisy_iq_norm")
os.makedirs(clean_dir, exist_ok=True)
os.makedirs(noisy_dir, exist_ok=True)

print("Save dir:")
print(" -", clean_dir)
print(" -", noisy_dir)

# =========================
# 유틸 함수
# =========================
def normalize_unit_mag(x, eps=1e-12):
    """frame 단위 |x| <= 1 (scale only; does not change SNR)."""
    m = np.max(np.abs(x))
    if m < eps:
        return x.astype(np.complex64)
    return (x / m).astype(np.complex64)

def add_awgn_complex(x, snr_db_bw, rng, *, bw_hz: float, fs_hz: float):
    """복소 AWGN 추가.

    IMPORTANT (OSF 보정):
    - 여기서 `snr_db_bw`는 'BW(=occupied bandwidth) 기준 SNR'로 해석합니다.
    - 샘플링이 fs=bw*OSF 일 때, 샘플 기준 SNR은
        SNR_sample(dB) = SNR_BW(dB) - 10*log10(OSF)
      이므로, 동일한 BW-SNR을 만들려면 샘플 노이즈 분산을 OSF만큼 더 키워야 합니다.

    참고: 마지막에 `normalize_unit_mag(y)`를 해도 스케일만 바뀌므로 SNR 자체는 유지되지만,
    clean/noisy가 서로 다른 스케일로 저장될 수 있어 학습(MSE)에는 불리할 수 있습니다.
    """
    osf = float(fs_hz) / float(bw_hz)
    snr_db_sample = float(snr_db_bw) - 10.0 * np.log10(osf)

    sig_power = float(np.mean(np.abs(x)**2) + 1e-12)
    snr_lin = 10.0 ** (snr_db_sample / 10.0)
    noise_power = sig_power / snr_lin
    sigma = np.sqrt(noise_power / 2.0)

    noise = (rng.normal(0, sigma, x.shape) + 1j * rng.normal(0, sigma, x.shape)).astype(np.complex64)
    y = (x.astype(np.complex64) + noise).astype(np.complex64)
    return y

# =========================
# 메타데이터 저장
# =========================
meta = {
    "sf": int(sf),
    "bw": float(bw),
    "OSF": int(OSF),
    "fs": int(fs),
    "N_symbols": int(N),
    "samples_per_symbol": int((2**sf) * OSF),
    "noisy_per_clean": int(NOISY_PER_CLEAN),
    "snr_db_range": [float(SNR_DB_LOW), float(SNR_DB_HIGH)],
    "snr_db_definition": "SNR over BW (not per-sample)",
    "note": "Clean + Noisy pairs saved. Training loads pairs from disk.",
}
with open(os.path.join(base_dir, "dataset_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Wrote:", os.path.join(base_dir, "dataset_meta.json"))

# =========================
# 데이터 생성 (Clean + Noisy 저장)
# =========================
rng = np.random.default_rng(42)
GENERATE = True
if GENERATE:
    print("Generating clean+noisy symbol pairs...")
    for sym in range(N):
        # 1) clean
        x_clean = lora.gen_symbol_fs(sym, sf=sf, bw=bw, Fs=fs).astype(np.complex64)
        x_gt = normalize_unit_mag(x_clean)
        np.save(os.path.join(clean_dir, f"sym_{sym:03d}_iq.npy"), x_gt)

        # 2) noisy (SNR: 0 ~ -30 dB over BW), 1~2회
        for k in range(NOISY_PER_CLEAN):
            snr_db = float(rng.uniform(SNR_DB_LOW, SNR_DB_HIGH))
            x_noisy = add_awgn_complex(x_gt, snr_db, rng, bw_hz=bw, fs_hz=fs)
            # 저장은 기존 정책 유지: frame 단위 |x|<=1로 스케일
            x_noisy = normalize_unit_mag(x_noisy)

            # 파일명에 sym + rep + snr 저장 (snr는 소수 1자리로 고정)
            np.save(
                os.path.join(noisy_dir, f"sym_{sym:03d}_rep{k:02d}_snr{snr_db:+05.1f}dB_iq.npy"),
                x_noisy,
            )

        if sym % 64 == 0:
            print(f"  {sym}/{N-1}")
    print("✅ Dataset generation done.")
else:
    print("Generation skipped.")

Loaded my_lora_utils from: /home/gwon9906/LoRa-bam-reconstruction/utils/my_lora_utils.py
